# obi-lookout quick start

Finds personal and business-sensitive spans (names, ID numbers, phones, emails, addresses, secrets, internal hosts...) in English, Malay and mixed text.

> **Use the sample text in this notebook only.** Anything you type into Colab is processed on Google's servers. For real text, run obi-lookout locally:
> see the [repository README](https://github.com/obiguard/obi-lookout-quickstart).
>
> It is **not a guarantee**: it misses some values and flags some that are not sensitive. Its published scores are on synthetic text.

Runtime: the free CPU is enough. Choose *Runtime > Run all*. The first run downloads the model (about 1.2 GB).

In [ ]:
# Colab already has torch. These are the other packages, at the versions the model was tested with.
!pip install -q gliner2==2.0.0 transformers==4.57.6 peft==0.21.2
!wget -q -O obi_lookout.py https://raw.githubusercontent.com/obiguard/obi-lookout-quickstart/main/obi_lookout.py

In [ ]:
from obi_lookout import ObiLookout

lookout = ObiLookout()   # downloads and loads the model once

## Find sensitive spans

In [ ]:
text = """Tolong hubungi Puan Aisha binti Rahman di aisha.rahman@syarikatmaju.test atau 012-345 6789 berkenaan projek Bintang Timur.
Nombor kad pengenalan beliau ialah 900231-14-5678 dan beliau tinggal di No. 12, Jalan Mawar 3, Taman Seri Indah, 47000 Petaling Jaya, Selangor."""

for s in lookout.detect(text):
    print(f"{s.label:<24} {s.text}   (confidence {s.score:.2f})")

## See them in the text

In [ ]:
import html
from IPython.display import HTML

COLOURS = {"person": "#ffd6a5", "email": "#caffbf", "phone": "#9bf6ff", "id_number": "#ffadad", "address": "#bdb2ff",
           "secret": "#ff9aa2", "internal_host": "#fdffb6", "organisation": "#a0c4ff", "client_or_project_name": "#ffc6ff"}

def show(text, spans):
    out, cursor = [], 0
    for s in spans:
        out.append(html.escape(text[cursor:s.start]))
        colour = COLOURS.get(s.label, "#dddddd")
        out.append(f'<mark style="background:{colour};color:#000;padding:1px 3px;border-radius:3px">{html.escape(s.text)}'
                   f'<sub style="font-size:70%"> {s.label}</sub></mark>')
        cursor = s.end
    out.append(html.escape(text[cursor:]))
    return HTML('<pre style="white-space:pre-wrap;line-height:1.8">' + "".join(out) + "</pre>")

show(text, lookout.detect(text))

## Redact them

In [ ]:
print(lookout.redact(text, "label"))   # [PERSON], [EMAIL] ...
print()
print(lookout.redact(text, "mask"))    # asterisks of the same length

## English, and mixed English and Malay

Edit the text and re-run. Keep it made-up.

In [ ]:
mixed = """Hi Farid, boleh you forward invoice untuk Project Kijang to Lim Wei Ming (wei.ming@contoh-corp.test)? Dia cakap bayaran SGD 3,900 masuk ke akaun 7012345678 minggu depan.
Kalau ada masalah, call dia di +60 11-2345 6789 atau check server jump-host-01.contoh.internal."""

show(mixed, lookout.detect(mixed))

## Next

- Run it locally, or as an OpenAI-compatible server in Docker: see the [README](https://github.com/obiguard/obi-lookout-quickstart).
- Limits, scores and how the model was built: the [model card](https://huggingface.co/obiguard/obi-lookout).